# Resource Builder Notebook

In [1]:
import warnings

from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import RES.RESources as RES_module

from RES import utility as utils
from RES.hdf5_handler import DataHandler

import RES.visual_styles as styles
style_path = Path(styles.__file__).parent / "elsevier.mplstyle"
plt.style.use(style_path)
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

cfg_path='config/config_CAN_policy1.yaml'
cfg=utils.load_config(cfg_path)


## Set Required Args to Initialize Objects

- All steps are integrated in this 'RES_module.build()' method.

In [2]:
# import RES.RESources as RES

# # Iterate over provinces for both solar and wind resources
# resource_types = ['wind','solar']  # 'solar'
# provinces=['BC']  #,'AB','SK','ON','NS','MB'
# for province_code in provinces:
#     for resource_type in resource_types:
#         required_args = {
#             "config_file_path": 'config/config_CAN.yaml',
#             "region_short_code": province_code,
#             "resource_type": resource_type
#         }
        
#         # Create an instance of Resources and execute the module
#         RES_module = RES.RESources_builder(**required_args)
        
    
#     # For complete workflow, uncomment the following line
#         # RES_module.build(select_top_sites=True,
#         #                  use_pypsa_buses=False)  


----

In [3]:
# Construct region_options as a list of tuples: (name, code)
region_options = [(cfg['region_mapping'][code]['name'], code) for code in cfg['region_mapping']]
region_code = 'BC'  # Default selection, change as needed

# Create dropdown widget for region codes with names shown, codes as values
region_code_dropdown = widgets.Dropdown(
    options=region_options,
    value=region_code,
    description='Region:',
    disabled=False,
)

display(region_code_dropdown)

Dropdown(description='Region:', index=1, options=(('Alberta', 'AB'), ('British Columbia', 'BC'), ('Manitoba', …

In [4]:
resource_type_dropdown = widgets.Dropdown(
    options=['wind', 'solar'],
    value='solar',
    description='Resource:',)
display(resource_type_dropdown)


Dropdown(description='Resource:', index=1, options=('wind', 'solar'), value='solar')

In [5]:
resource_type = resource_type_dropdown.value
region_code = region_code_dropdown.value



required_args = {
    "config_file_path": cfg_path,
    "region_short_code": region_code,
    "resource_type": resource_type
}


# Create an instance of Resources and execute the module
Builder = RES_module.RESources_builder(**required_args)

____________________________________________________________
     Initiating RESource Builder | RES.RESources
____________________________________________________________
  └─> RES.boundaries| Region Set to >> Short Code : BC, Name: British Columbia).
  └─> RES.boundaries| Collecting regional boundary...
  └─> RES.boundaries| Loading GADM boundaries (Sub-provincial | level =2) for British Columbia from local file data/processed_data/regions/gadm41_Canada_L2_BC.geojson.
  └─> RES.boundaries| Region Set to >> Short Code : BC, Name: British Columbia).
  └─> RES.boundaries| Collecting regional boundary...
  └─> RES.boundaries| Loading GADM boundaries (Sub-provincial | level =2) for British Columbia from local file data/processed_data/regions/gadm41_Canada_L2_BC.geojson.
 └> NREL_ATBProcessor initiated...
  └> RES.atb| Processing Annual Technology Baseline (ATB) data sourced from NREL...
 └> RES.utility| Directory 'data/downloaded_data/NREL/ATB/ATBe.parquet' found locally.
  └> RES.atb| ATB

In [ ]:
# Builder.clean_data_store()

# Stepwise Checks/Debugging [when required]

### Step 1: Prepare Spatial Grid Cells

- This method collects the sub-national administrative boundaries. 
- Using that boundary, we calculate the Minimum Bounding Rectangle (MBR). 
- We use that MBR as a cutout to source weather resources data from ERA5 via CDSAPI. The ERA5's cutout is then stored as a netcdf `.nc' file.
- We load that cutout as `atlite`'s `cutout` object.
- We then use `atlite`'s `cutout.grid` attribute to create our test beds for the analysis i.e. the grid cells (geodataframe)

In [ ]:
step1_results=Builder.get_grid_cells()

step1_results.head(5) # See the first 5 rows of the grid cells data

In [ ]:
# Builder.datahandler.refresh()
# Builder.datahandler.from_store('cells').head(5)

### Step 2: Calculate Potential Capacity

- This method loads the cutout (atlite's cutout object), regional boundary (GeoDataFrame), loads the cost parameters and  also initiates a __composite excluder__
  - The ([`atlite`'s exclusion container](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.availabilitymatrix)) to merge all the spatial layers.
- the `cutout.availabilitymatrix` method calculates % of usable area within each grid cell after applying exclusion criteria (e.g., protected areas, water bodies) and returns an [`AvaliabilityMatrix`](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.availabilitymatrix)
- We apply technology landuse intensity (e.g., MW/km² for wind or solar) to translate this to potential capacity data.
- We get the maximum installable capacity for each grid cell based on available area, land use constraints, and technology-specific parameters.
  > - Current results gives a percentage of availability for each grid cell. It does not tell specifically which spatial area inside a grid cell is unavailable.
  > - The _potential capacity_ translation processing involves `area` calculation. The area calculation method is integrated to `RES.cell_processor.get_capacity()`. That method is sensitive to area calculation specific coordinate-system projection of the geodataframe. It is recommended to be cautious about choosing this crs.
  

In [ ]:
step2_results=Builder.get_cell_capacity()

In [ ]:
raster_da=Builder.cell_processor.Availability_matrix

In [ ]:
# Builder.datahandler.refresh()
# Builder.datahandler.from_store('cells').head(5)

- The Landavailability Map size may need some adjustments to look nicer
  > the defatult setup is adjusted for BC map

In [ ]:
figure=Builder.cell_processor.plot_ERAF5_grid_land_availability(region_boundary=Builder.gadmBoundary.get_region_boundary(),
                                                            Availability_matrix=Builder.cell_processor.Availability_matrix,
                                                            figsize=(8, 8),
                                                            legend_box_x_y=(0.9, 0.9))


In [ ]:
# RES_module.cell_processor.Availability_matrix.plot(cmap='Greens')

### Step 3: Get CF and Windspeed from Higher Resolution Data

> - Currently configured for Wind Resources only. Wind resources (windspeed) are known to have significant variations across ERA5's ~30km resolution. We rescaled the windspeed with higher resolution windspeed from Global Wind Atlas (GWA). Then we calculate the ERA5 scaled windspeed from the mapped GWA cells. However, GWA does not provide hourly profiles. We source the profile from ERA5.

- returns NONE if result datafield ('windspeed_ERA5') is already there 

In [ ]:
gwa_cells=Builder.gwa_cells.load_gwa_cells()

* Plot Windspeed for 100m resolution cells (not a mandatory step, for cross checking purposes)
  > That's a high resolution data, may take a while to plot. Make sure your machine's cache memory is enough to hold this data.

In [ ]:
# gwa_cells.plot('windspeed_gwa',cmap="Blues",legend=True)

#### GWA Scaled Wind Speed vs ERA5 Windspeed Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))
sns.histplot(gwa_cells['windspeed_gwa'], bins=50, kde=True, color='steelblue', edgecolor='white', stat='density', alpha=0.6,legend=True)
sns.rugplot(gwa_cells['windspeed_gwa'], color='grey', height=0.02)

plt.title('Windspeed Distribution (GWA\'s 100m Resolution)', fontsize=14)
plt.xlabel('Windspeed (m/s)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f'vis/{region_code}/gwa_resolution_windspeed_distribution_{region_code}.jpg', dpi=300)
# plt.show()

In [ ]:
step3_results_A=Builder.extract_weather_data()

In [ ]:
step3_results_B=Builder.update_gwa_scaled_params() # testing, 2025 04 21

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

ax=Builder.datahandler.from_store('cells').plot('potential_capacity_wind',cmap='Blues',legend=True)
ax.patch.set_alpha(0)
ax.set_axis_off()

In [ ]:
ax=Builder.datahandler.from_store('cells').plot('potential_capacity_wind',cmap='Blues',legend=True)
ax.patch.set_alpha(0)
ax.set_axis_off()

* A comparison of ERA5 actual (from reanalysis dataset) windspeed vs GWA windspeed downscaled to ERA5 resolution.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Plot KDEs
step3_results_B['windspeed_ERA5'].plot.kde(color='orangered', linewidth=2, label='ERA5 Windspeed KDE')
step3_results_B['windspeed_gwa'].plot.kde(color='navy', linewidth=2, label='GWA Windspeed KDE')

# Plot histograms
step3_results_B['windspeed_ERA5'].plot.hist(bins=30, color='orange', edgecolor='white', density=True, alpha=0.4, label='ERA5 Windspeed')
step3_results_B['windspeed_gwa'].plot.hist(bins=30, color='skyblue', edgecolor='white', density=True, alpha=0.4, label='GWA Windspeed')

plt.title('Distribution of GWA and ERA5 Windspeed', fontsize=14, weight='bold')
plt.xlabel('Windspeed (m/s)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig(f'vis/{region_code}/ERA5_resolution_windspeed_distribution_ERA5vsGWA_{region_code}.png', dpi=300)
plt.show()

### Step 4: Get Timeseries

- We define technology attributes.
- We extract timeseries using weather resources data from ERA5's cutout.
  - The timeseries calculation method currently configured with [atlite.cutout.pv](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.pv) and [atlite.cutout.wind](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.wind) methods.

> __Attention__
  > - Configure the timezone conversion information carefully to ensure proper usage of the timeseries in downstream modelling. 
  > - ERA5 provides naive timezone index data. We use the timezone information from config file to enable the timezone shift of the timeseries.
  > - However, after conversion we removed the timezone awareness from the datetime index to harmonize with pypsa supported timeseries index.

In [ ]:
step4_results=Builder.get_CF_timeseries()

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

In [ ]:

ax=Builder.datahandler.from_store('cells').plot('wind_CF_mean',cmap='PuBu',legend=True)
ax.set_axis_off()

### Step 5: Find Grid Proximity

This information is critical for downstream operational analysis with this resource options.

> - Currently configured for Transmission Lines and/or Grid Substations.
> - We do not know the specific project point of a resource. Hence, the resource to grid-node distance has been calculated from the centroid of each grid to the grid node. 
> - If you have a specific project point, you should recalculate this distance with your specific project point.


- Identifies and assigns grid nodes to each cell. 
- Calculates distance (in km) from each grid cell to the nearest grid node (e.g., transmission line, substation) to assess connectivity and feasibility for energy transport.
    
    > If your use case of the resource options are to be plugged in to a downstream operational model (e.g. PyPSA), use harmonized nodes to populate this data.
    > harmonized nodes i.e. same data that are intended to be used as _bus_ nodes at your operational model. 


In [ ]:
step5_results=Builder.find_grid_nodes(use_lines=True)  # use_pypsa_buses=False

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

In [ ]:
ax=Builder.datahandler.from_store('cells').plot('nearest_station_distance_km',cmap='gray',legend=True)
ax.set_axis_off()

### Step 6: Scoring Metric to Rank the Sites

- Scores each grid cell based on multiple criteria (e.g., resource quality, proximity to grid), supporting site selection.

In [ ]:
step6_results=Builder.score_cells()

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

In [ ]:
Builder.datahandler.from_store('cells').lcoe_actualCap_wind.describe()

In [ ]:
Builder.datahandler.from_store('cells').lcoe_wind.describe()

In [ ]:
X=Builder.datahandler.from_store('cells')
X_f=X[X['potential_capacity_wind']>0]

In [ ]:
ax=X_f.plot('lcoe_wind',cmap='cool',legend=True)
ax.set_axis_off()

In [ ]:
# step6_results[step6_results['potential_capacity_wind']>0]

### Step 7: Clusterized Representation of the Sites

- Groups grid cells into clusters based on spatial or resource characteristics to enable aggregated analysis.
- Produces time series data for each cluster, summarizing the resource and capacity factor information at the cluster leve

In [ ]:
step7_results_Clusters=RES_module.get_clusters(
                                               wcss_tolerance=0.5,)
step7_results_ClusterTS=RES_module.get_cluster_timeseries()


In [ ]:
len(step7_results_Clusters[0])

In [ ]:
# step7_results_Clusters[0].head(10).plot()

- Creates Units Dictionary

In [ ]:
utils_dict=RES_module.units.create_units_dictionary()

# [Exploratory]

### Explore the outputs from Store

In [6]:
country_name=cfg.get('country','Canada') 
country_kwd=country_name.replace(' ','')

##  Define run/scenario

In [7]:
RUN_ID= cfg.get('Scenario').get('run_id')
store=f"data/store/resources_{country_kwd}_{region_code}_{RUN_ID}.h5"# f"../data/store/resources_{province_code}.h5" 
res_store=DataHandler(store,show_structure=True) # the DataHandler object could be initiated without the store definition as well.

____________________________________________________________
     RES.hdf5_handler|🗄️ Structure of HDF5 file: data/store/resources_Canada_BC_strict_policy_aeroway_CPCAD_buffer.h5
____________________________________________________________
[key] boundary
[key] cells
[key] clusters
  └─ [key] clusters/solar
  └─ [key] clusters/wind
[key] cost
  └─ [key] cost/atb
  └─   └─ [key] cost/atb/solar
  └─   └─ [key] cost/atb/wind
[key] dissolved_indices
  └─ [key] dissolved_indices/solar
  └─ [key] dissolved_indices/wind
[key] lines
[key] substations
[key] timeseries
  └─ [key] timeseries/clusters
  └─   └─ [key] timeseries/clusters/solar
  └─   └─ [key] timeseries/clusters/wind
  └─ [key] timeseries/solar
  └─ [key] timeseries/wind
[key] units


└> To access the data : 
 └> <datahandler instance>.from_store('<key>')


In [8]:
cells=res_store.from_store('cells')
boundary=res_store.from_store('boundary')
solar_clusters=res_store.from_store('clusters/solar')
wind_clusters=res_store.from_store('clusters/wind')
solar_clusters_ts=res_store.from_store('timeseries/clusters/solar')
wind_clusters_ts=res_store.from_store('timeseries/clusters/wind')
dissolved_indices_solar=res_store.from_store('dissolved_indices/solar')
dissolved_indices_wind=res_store.from_store('dissolved_indices/wind')

----
TEMP
----


In [9]:
cells

,x,y,Country,Province,Region,geometry,nearest_station,nearest_station_distance_km,Country_1,potential_capacity_wind,...,potential_capacity_solar,capex_solar,fom_solar,vom_solar,grid_connection_cost_per_km_solar,tx_line_rebuild_cost_solar,Operational_life_solar,solar_CF_mean,lcoe_solar,lcoe_actualCap_solar
cell,,,,,,,,,,,,,,,,,,,,,
EastKootenay_-116.0_50.5,-116.00,50.50,Canada,British Columbia,EastKootenay,"POLYGON ((-116.125 50.375, -116.125 50.625, -1...",BC_ATH_DSS,1.678665,Canada,89.616486,...,89.338311,1.366598,0.023766,0.0,2.6,0.56,25,0.233903,51.281155,5.144587e+01
EastKootenay_-115.75_49.5,-115.75,49.50,Canada,British Columbia,EastKootenay,"POLYGON ((-115.875 49.375, -115.875 49.625, -1...",BC_SPL_DSS,0.797887,Canada,746.598011,...,237.179679,1.366598,0.023766,0.0,2.6,0.56,25,0.228582,51.817953,5.169809e+01
EastKootenay_-115.0_50.25,-115.00,50.25,Canada,British Columbia,EastKootenay,"POLYGON ((-115.125 50.125, -115.125 50.375, -1...",BC_FRO_ISS,6.345190,Canada,342.943696,...,121.026073,1.366598,0.023766,0.0,2.6,0.56,25,0.243583,52.508526,5.233661e+01
Okanagan-Similkameen_-119.5_49.25,-119.50,49.25,Canada,British Columbia,Okanagan-Similkameen,"POLYGON ((-119.625 49.125, -119.625 49.375, -1...",BC_VAS_TSS,2.187432,Canada,321.851887,...,116.567965,1.366598,0.023766,0.0,2.6,0.56,25,0.228279,52.924391,5.287022e+01
Cariboo_-122.5_53.0,-122.50,53.00,Canada,British Columbia,Cariboo,"POLYGON ((-122.625 52.875, -122.625 53.125, -1...",BC_WFQ_ISS,0.459681,Canada,295.566985,...,263.722380,1.366598,0.023766,0.0,2.6,0.56,25,0.216765,54.376962,5.428338e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Bulkley-Nechako_-126.5_53.5,-126.50,53.50,Canada,British Columbia,Bulkley-Nechako,"POLYGON ((-126.625 53.375, -126.625 53.625, -1...",BC_HML_DSS,76.802924,Canada,0.000000,...,0.000000,1.366598,0.023766,0.0,2.6,0.56,25,0.206008,120.381876,9.999990e+05
Skeena-QueenCharlotte_-131.75_54.0,-131.75,54.00,Canada,British Columbia,Skeena-QueenCharlotte,"POLYGON ((-131.66396 54.125, -131.6706 54.1072...",BC_PRT_ISS,162.222373,Canada,0.000000,...,0.000000,1.366598,0.023766,0.0,2.6,0.56,25,0.158249,248.717315,9.999990e+05
Stikine_-138.5_60.0,-138.50,60.00,Canada,British Columbia,Stikine,"POLYGON ((-138.375 59.875, -138.625 59.875, -1...",BC_FKR_GSS,940.767821,Canada,0.000000,...,0.000000,1.366598,0.023766,0.0,2.6,0.56,25,0.177210,970.944340,9.999990e+05


---

- Interactive Map

In [ ]:
# wind_clusters[wind_clusters['lcoe']<=100].explore('potential_capacity')

# Playground for Top Site Selection

In [10]:
resource_clusters_solar,cluster_timeseries_solar=RES_module.select_top_sites(solar_clusters,
                                                                solar_clusters_ts,
                                                                    resource_max_capacity=10)

resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
                                                                wind_clusters_ts,
                                                                    resource_max_capacity=50)

>>> Selecting TOP Sites to for 10 GW Capacity Investment in BC...
____________________________________________________________________________________________________
Selecting the Top Ranked Sites to invest in 10 GW PV in BC
____________________________________________________________________________________________________

!! Note: The Last cluster (Thompson-Nicola_1) originally had 4.53 GW potential capacity.To fit the maximum capacity investment of 10 GW, it has been adjusted to 2.46 GW

>>> Selecting TOP Sites to for 50 GW Capacity Investment in BC...
____________________________________________________________________________________________________
Selecting the Top Ranked Sites to invest in 50 GW PV in BC
____________________________________________________________________________________________________

!! Note: The Last cluster (Thompson-Nicola_2) originally had 11.54 GW potential capacity.To fit the maximum capacity investment of 50 GW, it has been adjusted to 6.95 GW



In [ ]:
dissolved_indices_wind

In [ ]:
RES_module.export_results('wind',
                          province_code,
                    resource_clusters_wind,
                    cluster_timeseries_wind,)

In [ ]:
RES_module.export_results('solar',
                             province_code,
                    resource_clusters_solar,
                    cluster_timeseries_solar,)

In [ ]:
resource_clusters_solar.plot('potential_capacity',legend=True)
resource_clusters_wind.plot('potential_capacity',legend=True)